# Audio Window-Level Feature Extraction — GeMAPSv01b

Extracts **opensmile GeMAPSv01b** acoustic features (62 parameters) per **30-second window** per participant, then group-averages across P1–P4 to produce one row per `group_id × task_id × window_index`.

This extends `tools/audio_feature_extraction_egemaps.py`, which produces only task-level summaries, to the window granularity needed for the HMM.

## Why this matters
The `feature_inventory.md` identifies 4 audio features that would be the most valuable additions to the HMM if windowed:
- `audio_energy_mean` — vocal arousal (group mean)
- `audio_pitch_mean` — emotional tone (unique signal)
- `audio_pitch_sd` — expressiveness / intonation range
- `audio_hnr_mean` — vocal stress / voice clarity

## Input
- WAV files: per-speaker DPA 4060 recordings, one file per `participant × task`
  - Pattern: `*_task-{T1,T2,T3}_*_acq-dpa_mic{9,10,11,12}_aud.wav`
- Window metadata: `icmi_paper/results/hmm_input_features_final.tsv`
  - Provides `group_id, task_id, window_index, window_start_s` for alignment

## Output
- `icmi_paper/results/audio_window_features.tsv` — one row per `group_id × task_id × window_index`
- Columns: metadata + 4 priority features + full 62-feature GeMAPSv01b (prefixed `gemap_`)

## Required environment
Needs the `affectai-capture` venv which has `opensmile` and `audiofile`:
```bash
# From repo root
conda activate affectai-capture  # or use the .venv
jupyter notebook icmi_paper/analysis/audio_window_features.ipynb
```

In [1]:
import logging
import re
import warnings
from pathlib import Path

import numpy as np
import opensmile
import pandas as pd

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
log = logging.getLogger(__name__)

print(f"opensmile {opensmile.__version__}")


opensmile 2.6.0


In [2]:
# ── Paths — adjust AUDIO_ROOT to where the DPA WAV files live ─────────────────
REPO_ROOT   = Path('../..')          # affectai-data-processing/
RESULTS_DIR = Path('../results')
AUDIO_ROOT  = Path('C:/Users/amodica/Downloads/audio files')

# ── Window parameters (must match HMM features) ───────────────────────────────
WINDOW_S = 30.0    # window duration in seconds

# ── Mic → participant mapping (consistent across all groups) ──────────────────
# Privacy: participant IDs are P1–P4 only; no real names stored.
MIC_TO_PARTICIPANT = {'mic9': 'P1', 'mic10': 'P2', 'mic11': 'P3', 'mic12': 'P4'}

# ── Tasks to extract (T0 excluded — no HMM windows for baseline) ─────────────
TARGET_TASKS = {'T1', 'T2', 'T3'}

# ── 4 priority features (column names in GeMAPSv01b output) ──────────────────
PRIORITY_FEATURES = {
    'loudness_sma3_amean':                          'audio_energy_mean',
    'F0semitoneFrom27.5Hz_sma3nz_amean':            'audio_pitch_mean',
    'F0semitoneFrom27.5Hz_sma3nz_stddevNorm':       'audio_pitch_sd',
    'HNRdBACF_sma3nz_amean':                        'audio_hnr_mean',
}

# ── Regex to parse WAV filename ───────────────────────────────────────────────
# sub-01_ses-20260312_grp-07_run01_task-T2_run-01_acq-dpa_mic9_aud.wav
_WAV_RE = re.compile(
    r'ses-(?P<ses_date>[0-9]{8})_'
    r'(?P<grp>grp-[0-9]+)_'
    r'run(?P<ses_run>[0-9]+)'
    r'.*?task-(?P<task>T[0-9]+)'
    r'.*?dpa-(?P<mic>mic[0-9]+)-aud'
    r'.*?_audio[.]wav$',
    re.IGNORECASE,
)

print('Config OK')
print(f'  Audio root: {AUDIO_ROOT}')
print(f'  Window size: {WINDOW_S}s')

Config OK
  Audio root: C:\Users\amodica\Downloads\audio files
  Window size: 30.0s


## Step 1: Load existing window metadata

We use the HMM input features file to know exactly which `(group_id, task_id, window_index, window_start_s)` combinations exist. This ensures audio windows are perfectly aligned with the physio windows.

In [3]:
hmm_feats = pd.read_csv(RESULTS_DIR / 'hmm_input_features_final.tsv', sep='\t')
windows = (
    hmm_feats[['group_id', 'task_id', 'window_index', 'window_start_s']]
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f'Windows to extract: {len(windows)}')
print(f'Groups: {sorted(windows["group_id"].unique())}')
print(f'Tasks:  {sorted(windows["task_id"].unique())}')
print(f'Window indices: {windows["window_index"].min()} – {windows["window_index"].max()}')
windows.head()

Windows to extract: 526
Groups: ['grp-07', 'grp-08', 'grp-09', 'grp-10', 'grp-11', 'grp-12', 'grp-13', 'grp-14', 'grp-15', 'grp-16']
Tasks:  ['T1', 'T2', 'T3']
Window indices: 0 – 33


,group_id,task_id,window_index,window_start_s
0,grp-07,T1,0,0.0
1,grp-07,T1,1,30.0
2,grp-07,T1,2,60.0
3,grp-07,T1,3,90.0
4,grp-07,T1,4,120.0


## Step 2: Discover WAV files

In [4]:
def discover_wavs(audio_root: Path, target_tasks: set) -> dict:
    """Return a dict keyed by (group_id, task_id, participant_id) → wav_path.

    Excludes processed variants (_hp, _hp_cal, _hp_norm) and peaks subdirectories.
    """
    wav_map = {}
    for wav in sorted(audio_root.rglob('*dpa-mic*-aud*_audio.wav')):
        if 'peaks' in wav.parts:
            continue
        if re.search(r'_aud_hp', wav.name, re.IGNORECASE):
            continue
        m = _WAV_RE.search(wav.name)
        if not m:
            continue
        task = m.group('task').upper()
        if task not in target_tasks:
            continue
        grp  = m.group('grp')
        mic  = m.group('mic')
        part = MIC_TO_PARTICIPANT.get(mic, mic)
        key  = (grp, task, part)
        if key in wav_map:
            log.warning('Duplicate key %s — keeping first: %s', key, wav_map[key].name)
        else:
            wav_map[key] = wav
    return wav_map


wav_map = discover_wavs(AUDIO_ROOT, TARGET_TASKS)
print(f'Found {len(wav_map)} WAV files')

# Summarise coverage
from collections import Counter
by_task = Counter(task for (_, task, _) in wav_map)
by_grp  = Counter(grp  for (grp,  _, _) in wav_map)
print('By task:', dict(sorted(by_task.items())))
print('By group:', dict(sorted(by_grp.items())))

Found 116 WAV files
By task: {'T1': 40, 'T2': 40, 'T3': 36}
By group: {'grp-07': 12, 'grp-08': 12, 'grp-09': 12, 'grp-10': 12, 'grp-11': 12, 'grp-12': 12, 'grp-13': 8, 'grp-14': 12, 'grp-15': 12, 'grp-16': 12}


## Step 3: Window extraction function

For each 30-second window we:
1. Load the WAV audio slice with `audiofile` (avoids loading the entire file into memory)
2. Feed the signal to opensmile's `process_signal()` to get 62 GeMAPSv01b functionals

If a window extends beyond the file end (last partial window), we pad with silence.

In [5]:
# Build the opensmile extractor once (expensive to initialise)
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.GeMAPSv01b,
    feature_level=opensmile.FeatureLevel.Functionals,
    verbose=False,
)
GEMALPS_COLS = smile.feature_names   # 62 column names
print(f'GeMAPSv01b features: {len(GEMALPS_COLS)}')
print('First 5:', GEMALPS_COLS[:5])

GeMAPSv01b features: 62
First 5: ['F0semitoneFrom27.5Hz_sma3nz_amean', 'F0semitoneFrom27.5Hz_sma3nz_stddevNorm', 'F0semitoneFrom27.5Hz_sma3nz_percentile20.0', 'F0semitoneFrom27.5Hz_sma3nz_percentile50.0', 'F0semitoneFrom27.5Hz_sma3nz_percentile80.0']


In [6]:
def extract_window_features(
    wav_path: Path,
    start_s: float,
    window_s: float = WINDOW_S,
) -> "pd.Series | None":
    """Extract 62 GeMAPSv01b functionals for one 30-second window.

    opensmile >= 2.x supports start/end float parameters directly in process_file.
    Returns a pd.Series of feature values, or None on failure.
    """
    end_s = start_s + window_s
    try:
        feats = smile.process_file(str(wav_path), start=start_s, end=end_s)
    except Exception as exc:
        log.warning("opensmile failed start=%.1fs in %s: %s", start_s, wav_path.name, exc)
        return None

    if feats is None or feats.empty:
        return None

    return feats.iloc[0]  # 62-element Series


print("Function defined OK")
print(f"  30s windows via smile.process_file(start=, end=)  [opensmile {opensmile.__version__}]")


Function defined OK
  30s windows via smile.process_file(start=, end=)  [opensmile 2.6.0]


## Step 4: Main extraction loop

For each `(group_id, task_id, window_index)` in the HMM windows, extract features for each of P1–P4.

Progress is saved incrementally to `_audio_window_partial.tsv` so extraction can be resumed if interrupted.

In [7]:
PARTIAL_PATH = RESULTS_DIR / '_audio_window_partial.tsv'
PARTICIPANTS = ['P1', 'P2', 'P3', 'P4']

# Resume from partial if it exists
if PARTIAL_PATH.exists():
    done_df = pd.read_csv(PARTIAL_PATH, sep='\t')
    done_keys = set(zip(done_df['group_id'], done_df['task_id'],
                        done_df['window_index'], done_df['participant_id']))
    rows = done_df.to_dict('records')
    print(f'Resuming from {len(done_df)} already-extracted rows')
else:
    done_keys = set()
    rows = []
    print('Starting fresh extraction')

total = len(windows) * len(PARTICIPANTS)
n_done = len(rows)
n_skip = 0
n_fail = 0

for _, win in windows.iterrows():
    grp        = win['group_id']
    task       = win['task_id']
    win_idx    = int(win['window_index'])
    start_s    = float(win['window_start_s'])

    for part in PARTICIPANTS:
        key = (grp, task, win_idx, part)
        if key in done_keys:
            continue

        wav_path = wav_map.get((grp, task, part))
        if wav_path is None:
            n_skip += 1
            continue

        feats = extract_window_features(wav_path, start_s)

        row = {
            'group_id':      grp,
            'task_id':       task,
            'window_index':  win_idx,
            'window_start_s': start_s,
            'participant_id': part,
        }
        if feats is not None:
            # Prefix all opensmile columns with gemap_ to avoid name clashes
            row.update({f'gemap_{k}': v for k, v in feats.items()})
        else:
            # Fill with NaN so the row still exists for group-averaging
            row.update({f'gemap_{k}': np.nan for k in GEMALPS_COLS})
            n_fail += 1

        rows.append(row)
        n_done += 1

        # Save checkpoint every 100 rows
        if n_done % 100 == 0:
            pd.DataFrame(rows).to_csv(PARTIAL_PATH, sep='\t', index=False)
            pct = 100 * n_done / total
            print(f'  {n_done}/{total} ({pct:.0f}%)  skip={n_skip}  fail={n_fail}')

participant_df = pd.DataFrame(rows)
participant_df.to_csv(PARTIAL_PATH, sep='\t', index=False)
print(f'\nExtraction complete: {len(participant_df)} participant-level rows')
print(f'  skipped (no WAV): {n_skip}  |  failed (opensmile): {n_fail}')
participant_df.head(3)

EmptyDataError: No columns to parse from file

## Step 5: Group-average across P1–P4

For each `(group_id, task_id, window_index)` compute the mean of each feature across the 4 participants, matching the convention used for physio features in the HMM.

In [ ]:
gemap_cols = [c for c in participant_df.columns if c.startswith('gemap_')]

group_df = (
    participant_df
    .groupby(['group_id', 'task_id', 'window_index', 'window_start_s'])[gemap_cols]
    .mean()         # mean across ≤4 participants; NaN-safe (skipna=True by default)
    .reset_index()
)

print(f'Group-level rows: {len(group_df)}')
print(f'Expected:         {len(windows)} (one per window)')
print(f'Missing windows:  {len(windows) - len(group_df)}')
group_df.head(3)

## Step 6: Add short-name priority columns and save

In [ ]:
# Add human-readable aliases for the 4 priority features
for gemal_col, short_name in PRIORITY_FEATURES.items():
    full_col = f'gemap_{gemal_col}'
    if full_col in group_df.columns:
        group_df[short_name] = group_df[full_col]
    else:
        print(f'WARNING: column {full_col} not found — check GeMAPSv01b column names')

# Reorder: metadata + priority features first, then full gemap_ columns
meta_cols     = ['group_id', 'task_id', 'window_index', 'window_start_s']
priority_cols = list(PRIORITY_FEATURES.values())
out_df = group_df[meta_cols + priority_cols + gemap_cols]

out_path = RESULTS_DIR / 'audio_window_features.tsv'
out_df.to_csv(out_path, sep='\t', index=False)
print(f'Saved: {out_path}')
print(f'Shape: {out_df.shape}')

# Coverage report for priority features
print('\nCoverage of priority features (non-NaN %):',
      (out_df[priority_cols].notna().mean() * 100).round(1).to_dict())

## Step 7: Merge with HMM features and quick sanity check

In [ ]:
merged = hmm_feats.merge(
    out_df[meta_cols + priority_cols],
    on=['group_id', 'task_id', 'window_index'],
    how='left',
)
print(f'Merged shape: {merged.shape}')
print('Coverage after merge:')
for col in priority_cols:
    pct = merged[col].notna().mean() * 100
    print(f'  {col}: {pct:.1f}%')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.0)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
feature_labels = {
    'audio_energy_mean': 'Loudness (energy mean)',
    'audio_pitch_mean':  'Pitch F0 (semitone, voiced)',
    'audio_pitch_sd':    'F0 variability (stddevNorm)',
    'audio_hnr_mean':    'HNR dBACF (voice clarity)',
}

for ax, col in zip(axes.flatten(), priority_cols):
    task_means = merged.groupby('task_id')[col].mean()
    ax.bar(task_means.index, task_means.values, color=['#e74c3c', '#3498db', '#2ecc71'])
    ax.set_title(feature_labels.get(col, col), fontsize=10, fontweight='bold')
    ax.set_xlabel('Task')
    ax.grid(axis='y', alpha=0.4)

plt.suptitle('Mean audio features per task (window-level, group-averaged)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/audio_window_task_means.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sanity check: task-level means')

## Step 8: Correlation with existing HMM features

Check whether the audio features are genuinely independent from the physio/speech features already in the HMM. If |r| < 0.4 with all existing features, they add new information.

In [ ]:
import numpy as np

HMM_FEATS = [
    'group_hr_mean_bpm_mean', 'group_eda_phasic_rate_hz_mean',
    'group_temp_mean_mean', 'group_et_pupil_mean_mean',
    'tr_silence_duration_s', 'tr_backchannel_count', 'tr_laughter_count',
    'tr_n_active_speakers', 'tr_ovl_count', 'tr_ovl_contested',
    'tr_ovl_time_s', 'tr_ovl_collaboration_index',
]

print('Spearman correlations: audio priority features vs HMM features')
print('(computed on windows where all features are non-NaN)\n')
print(f'{"Feature":<30}', end='')
for hmm_f in HMM_FEATS:
    short = hmm_f.replace('group_', '').replace('tr_', '')[:12]
    print(f'{short:>14}', end='')
print()

from scipy.stats import spearmanr
for audio_f in priority_cols:
    print(f'{audio_f:<30}', end='')
    for hmm_f in HMM_FEATS:
        if hmm_f not in merged.columns or audio_f not in merged.columns:
            print(f'{"N/A":>14}', end='')
            continue
        valid = merged[[audio_f, hmm_f]].dropna()
        if len(valid) < 10:
            print(f'{"<10":>14}', end='')
        else:
            r, _ = spearmanr(valid[audio_f], valid[hmm_f])
            marker = '**' if abs(r) > 0.6 else ('*' if abs(r) > 0.4 else '')
            print(f'{r:>11.2f}{marker:>3}', end='')
    print()